In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time
import math
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC 

In [ ]:
# вариант 1 (Мишин генератор)
data = np.loadtxt('data_model/linData_1000_10_09.txt')
Y_full = data[:, 0]
X_full = data[:, 1:]


In [2]:
# вариант 2 (генератор DataGeneration2)
#data = np.load('data_model/dg2_lin_1000_10_02_08.npz')
#data = np.load('data_model/dg2_lin_300k_5k_b0_02_08.npz')
data = np.load('data_model/dg2_lin_120k_5k_08z_b0_02_08.npz')
#data = np.load('data_model/dg2_lin_20_10_02_08.npz')
ss=StandardScaler()
X_full = ss.fit_transform(data["X"])
#X_full = data["X"]
Y_full = data["Y"]


In [7]:
from sklearn.model_selection import KFold
def GetKFoldsIndexes(n_indexes, n_splits, random_state=42, shuffle = True):
    # разбивает множество индексов от 0 до n_indexes-1 на n_splits частей 
    # shuffle - перемешивать индексы или нет  
    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle = True)
    inds = range(n_indexes)
    kf.get_n_splits(inds)
    isplit = kf.get_n_splits(inds)
    folds = []
    for i, ids in enumerate(kf.split(inds)):
        folds.append(ids[1])
    return folds    
    
def TrainLinearSVM(X, Y, ker = 'linear'):
    t_start = time.time()
    model = SVC(kernel=ker)
    model.fit(X, Y)
    res = {}
    res['a'] = model.coef_
    res['b'] = model.intercept_
    #res['nSV'] = model.
    t_end = time.time()
    res['time'] = t_end - t_start
    return res

def TrainLinearSVM_v2(X, Y, model):
    t_start = time.time()
    model.fit(X, Y)
    res = {}
    res['a'] = model.coef_[0]
    res['b'] = model.intercept_[0]
    #res['nSV'] = model.
    t_end = time.time()
    res['time'] = t_end - t_start
    return res
    
def TestLinearSVM_sub(X,Y, inds, model, prn = False):
    # Тестирование линейного svm в подпространстве inds
    # X, Y - полный набор данных 
    # inds - индексы используемых признаков
    # model['a'] - направляющий вектор в пространстве model['inds'] признаков 
    # model['b'] - смещение 
    t_start = time.time()
    scores = np.dot(X[:,inds], np.transpose(model['a'])[:,0])+model['b']
    miscl_neg = np.sum((Y == -1) & (scores > 0))
    miscl_pos = np.sum((Y == +1) & (scores < 0))
    #print(miscl_neg, miscl_pos)
    acc = 1 - (miscl_neg + miscl_pos) / len(Y)
    t_end = time.time()
    return scores, miscl_pos, miscl_neg, acc, t_end-t_start 

def TestLinearSVM(X,Y, model):
    # Тестирование линейного svm в подпространстве inds
    # X, Y - полный набор данных 
    # model['a'] - направляющий вектор в пространстве model['inds'] признаков 
    # model['b'] - смещение 
    t_start = time.time()
    scores = np.dot(X, np.transpose(model['a'])[:,0])+model['b']
    miscl_neg = np.sum((Y == -1) & (scores > 0))
    miscl_pos = np.sum((Y == +1) & (scores < 0))
    acc = 1 - (miscl_neg + miscl_pos) / len(Y)
    t_end = time.time()
    return acc, t_end-t_start

def TestLinearSVM_v2(X,Y, model):
    # Тестирование линейного svm в подпространстве inds
    # X, Y - полный набор данных 
    # model['a'] - направляющий вектор в пространстве model['inds'] признаков 
    # model['b'] - смещение 
    t_start = time.time()
    scores = np.dot(X, np.transpose(model['a']))+model['b']
    miscl_neg = np.sum((Y == -1) & (scores > 0))
    miscl_pos = np.sum((Y == +1) & (scores < 0))
    acc = 1 - (miscl_neg + miscl_pos) / len(Y)
    t_end = time.time()
    return acc, t_end-t_start
    
def TrainLinearSVMs_sub_v2(folds, X, Y, model, prn=False):
    tr_folds = []
    nfolds = len(folds)
    tr_time = 0
    for i in range(nfolds): 
        temp = TrainLinearSVM_v2(X[:,folds[i]], Y, model)
        temp['inds'] = np.array(folds[i])
        tr_folds.append(temp)
        if prn:
            print('sub ', i, ': time=', temp['time']) 
        tr_time = tr_time + temp['time']
        #if prn:
        #    for i in range(nfolds): 
        #        print(i, tr_folds[i])
    return tr_folds, tr_time

def CombLinearSVMs(tr_res):
# tr_res - список результатов обучения на подпространствах
# tr_res[i]['inds'] - индексы признаков, вошедших в подпространство
# tr_res[i]['a'] - направляющий вектор 
# tr_res[i]['b'] - смещение
    time_start = time.time()
    nfolds = len(tr_res)
    a_comb = tr_res[0]['a']
    b_comb = tr_res[0]['b']
    inds_comb = tr_res[0]['inds']
    for i in range(1, nfolds):
        a_comb = np.hstack((a_comb, tr_res[i]['a']))
        b_comb = b_comb + tr_res[i]['b']
        inds_comb = np.hstack((inds_comb, tr_res[i]['inds'])) 
    b_comb = b_comb/nfolds    
    comb = {}
    comb['a'] = a_comb
    comb['b'] = b_comb
    comb['inds'] = inds_comb
    time_end = time.time()
    comb['time'] = time_end - time_start
    return comb 

def TrainAndTestLinearSVM_AllSub(Xtr, Ytr, Xtst, Ytst, reps, prn=True):
    tr_res = []
    nrep = len(reps)
    for j in range(nrep):
        temp = {}
        temp['sub'], tr_time_subs = TrainLinearSVMs_sub(reps[j], Xtr, Ytr, ker='linear', prn = False) 
        tr_res.append(temp)
    #print('ok')
    for j in range(nrep):
        print(f" ---- rep { j+1} --------")
        for i in range(len(reps[j])):
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tr'] = acc
            tr_res[j]['sub'][i]['tst_time_tr'] = tst_time
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tst'] = acc
            tr_res[j]['sub'][i]['tst_time_tst'] = tst_time
            if prn:
                # печать с индексами
                #print(f"i={i}, inds={tr_res[j]['sub'][i]['inds']} tr_time={tr_res[j]['sub'][i]['tr_time]:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst={tr_res[j]['sub'][i]['acc_tst']:.4f}")
                # печать без индексов
                print(f"subspace {i}:  tr_time={tr_res[j]['sub'][i]['time']:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst= {tr_res[j]['sub'][i]['acc_tst']:.4f}")
                print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")

        tr_res[j]['comb'] = CombLinearSVMs(tr_res[j]['sub'])
        tr_res[j]['comb']['tr_time'] = tr_time_subs
        tr_res[j]['comb']['full_time'] = tr_res[j]['comb']['time'] + tr_time_subs

        _, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tr'] = acc_tr
        tr_res[j]['comb']['acc_tst_time_tr'] = tst_time_tr 

        _, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tst'] = acc_tst
        tr_res[j]['comb']['acc_tst_time_tst'] = tst_time_tst
        if prn:
            print(f"Комбинированная гиперплоскость:  время обучения: {tr_res[j]['comb']['tr_time']:.4f}, время объединения: {tr_res[j]['comb']['time']:.4f}, полное время: {tr_res[j]['comb']['full_time']:.4f} ")
            print(f"acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            #print(f"comb: inds={tr_res[j]['comb']['inds']} acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            
            # время тестирования: 
            #print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")
    return tr_res

def TrainAndTestLinearSVM_AllSub_v2(Xtr, Ytr, Xtst, Ytst, reps, model, prn=True):
    tr_res = []
    nrep = len(reps)
    for j in range(nrep):
        temp = {}
        temp['sub'], tr_time_subs = TrainLinearSVMs_sub_v2(reps[j], Xtr, Ytr, model, prn = False) 
        tr_res.append(temp)
    #print('ok')
    for j in range(nrep):
        if (prn):
            print(f" ---- rep { j+1} --------")
        for i in range(len(reps[j])):
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tr'] = acc
            tr_res[j]['sub'][i]['tst_time_tr'] = tst_time
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tst'] = acc
            tr_res[j]['sub'][i]['tst_time_tst'] = tst_time
            if prn:
                # печать с индексами
                #print(f"i={i}, inds={tr_res[j]['sub'][i]['inds']} tr_time={tr_res[j]['sub'][i]['tr_time]:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst={tr_res[j]['sub'][i]['acc_tst']:.4f}")
                # печать без индексов
                print(f"subspace {i}:  tr_time={tr_res[j]['sub'][i]['time']:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst= {tr_res[j]['sub'][i]['acc_tst']:.4f}")
                print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")

        tr_res[j]['comb'] = CombLinearSVMs(tr_res[j]['sub'])
        tr_res[j]['comb']['tr_time'] = tr_time_subs
        tr_res[j]['comb']['full_time'] = tr_res[j]['comb']['time'] + tr_time_subs

        _, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tr'] = acc_tr
        tr_res[j]['comb']['acc_tst_time_tr'] = tst_time_tr 

        _, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tst'] = acc_tst
        tr_res[j]['comb']['acc_tst_time_tst'] = tst_time_tst
        if prn:
            print(f"Комбинированная гиперплоскость:  время обучения: {tr_res[j]['comb']['tr_time']:.4f}, время объединения: {tr_res[j]['comb']['time']:.4f}, полное время: {tr_res[j]['comb']['full_time']:.4f} ")
            print(f"acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            #print(f"comb: inds={tr_res[j]['comb']['inds']} acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            
            # время тестирования: 
            #print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")
    return tr_res

def ConstractCombinedHyperplane(Xtr, Ytr, model, n_splits):
    n_obj, n_feat = np.shape(Xtr)
    state = int(time.time()*10000%1000)
    time_start = time.time()
    tr_res = dict()
    folds = GetKFoldsIndexes(n_feat, n_splits=n_splits, random_state=state, shuffle = False)
    tr_res['sub'], tr_subs_time = TrainLinearSVMs_sub_v2(folds, Xtr, Ytr, model, prn=False)
    tr_res['comb'] = CombLinearSVMs(tr_res['sub'])
    time_end = time.time() 
    tr_res['comb']['full_time'] = time_end - time_start
    tr_res['comb']['tr_time'] = tr_subs_time
    return tr_res 

  

def TrainAndTestLinearSVM_AllSub_v3(Xtr, Ytr, Xtst, Ytst, model, n_splits, prn=True):
    tr_res = []
    nrep = len(reps)
    for j in range(nrep):
        temp = {}
        
        temp['sub'], tr_time_subs = TrainLinearSVMs_sub_v2(reps[j], Xtr, Ytr, model, prn = False) 
        tr_res.append(temp)
    #print('ok')
    for j in range(nrep):
        if (prn):
            print(f" ---- rep { j+1} --------")
        for i in range(len(reps[j])):
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tr'] = acc
            tr_res[j]['sub'][i]['tst_time_tr'] = tst_time
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tst'] = acc
            tr_res[j]['sub'][i]['tst_time_tst'] = tst_time
            if prn:
                # печать с индексами
                #print(f"i={i}, inds={tr_res[j]['sub'][i]['inds']} tr_time={tr_res[j]['sub'][i]['tr_time]:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst={tr_res[j]['sub'][i]['acc_tst']:.4f}")
                # печать без индексов
                print(f"subspace {i}:  tr_time={tr_res[j]['sub'][i]['time']:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst= {tr_res[j]['sub'][i]['acc_tst']:.4f}")
                print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")

        tr_res[j]['comb'] = CombLinearSVMs(tr_res[j]['sub'])
        tr_res[j]['comb']['tr_time'] = tr_time_subs
        tr_res[j]['comb']['full_time'] = tr_res[j]['comb']['time'] + tr_time_subs

        _, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tr'] = acc_tr
        tr_res[j]['comb']['acc_tst_time_tr'] = tst_time_tr 

        _, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tst'] = acc_tst
        tr_res[j]['comb']['acc_tst_time_tst'] = tst_time_tst
        if prn:
            print(f"Комбинированная гиперплоскость:  время обучения: {tr_res[j]['comb']['tr_time']:.4f}, время объединения: {tr_res[j]['comb']['time']:.4f}, полное время: {tr_res[j]['comb']['full_time']:.4f} ")
            print(f"acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            #print(f"comb: inds={tr_res[j]['comb']['inds']} acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            
            # время тестирования: 
            #print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")
    return tr_res


In [8]:
#X_train5k,X_test5k,Y_train5k,Y_test5k =train_test_split(X_full,Y_full,train_size = 5000, test_size=5000, random_state=42)
X_train10k, X_test10k,Y_train10k,Y_test10k =train_test_split(X_full,Y_full,train_size = 10000, test_size=10000, random_state=42)
#X_train90k, X_test10k,Y_train90k,Y_test10k =train_test_split(X_full,Y_full,train_size = 90000, test_size=10000, random_state=42)
X_train100k, X_test10k,Y_train100k,Y_test10k =train_test_split(X_full,Y_full,train_size = 100000, test_size=10000, random_state=42)
#X_train200k, X_test10k,Y_train200k,Y_test10k =train_test_split(X_full,Y_full,train_size = 200000, test_size=10000, random_state=42)

In [25]:
X_train1k, X_test1k,Y_train1k,Y_test1k =train_test_split(X_full,Y_full,train_size = 1000, test_size=1000, random_state=42)

In [10]:
# Объединение гиперплоскостей 
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test10k
Ytst = Y_test10k 
thr = 0.001
nreps = 3
tr_times = np.zeros((nreps))
tr_accs = np.zeros((nreps))
tst_accs = np.zeros((nreps))
splits = [5, 10, 20, 50, 100, 200, 500, 1000]
cs = [0.005, 0.01, 0.05, 0.1, 0.25, 0.5]  
for C in cs:     
    for n_splits in splits:
        print('------- C=',C,' n_splits=',n_splits,' ------------')
        for rep in range(nreps):
            tr_res = dict()
            model = LinearSVC(C=C, penalty="l1", dual=False)
            tr_res = ConstractCombinedHyperplane(Xtr, Ytr, model=model, n_splits=n_splits)
            big_a = (abs(tr_res['comb']['a'])>thr)
            comb_inds = tr_res['comb']['inds']
            large_inds = comb_inds[big_a]
            short_model=dict()
            short_model['a'] = tr_res['comb']['a'][big_a]
            short_model['b'] = tr_res['comb']['b']
            short_model['inds'] = large_inds
            acc_tr, _ = TestLinearSVM_v2(Xtr[:,large_inds],Ytr, short_model) 
            acc_tst, _ = TestLinearSVM_v2(Xtst[:,large_inds],Ytst, short_model)
            tr_times[rep] = tr_res['comb']['full_time']
            tr_accs[rep] = acc_tr
            tst_accs[rep] = acc_tst
            print(f" rep {rep}:  tr_time={tr_times[rep]:.4f} acc_tr={acc_tr:.4f}, acc_tst={acc_tst:.4f}, n_large_cf={len(large_inds)}")
        print(f"tr_time: [mean={np.mean(tr_times):.4f}, std={np.std(tr_times):.4f}],  acc_tr: [mean={np.mean(tr_accs):.4f}, std={np.std(tr_accs):.4f}], acc_tst: [mean={np.mean(tst_accs):.4f}, std={np.std(tst_accs):.4f}]")
    

------- C= 0.005  n_splits= 5  ------------
 rep 0:  tr_time=3.1507 acc_tr=0.9205, acc_tst=0.8987, n_large_cf=1614
 rep 1:  tr_time=3.2366 acc_tr=0.9204, acc_tst=0.8985, n_large_cf=1617
 rep 2:  tr_time=3.0168 acc_tr=0.9200, acc_tst=0.8981, n_large_cf=1646
tr_time: [mean=3.1347, std=0.0905],  acc_tr: [mean=0.9203, std=0.0002], acc_tst: [mean=0.8984, std=0.0002]
------- C= 0.005  n_splits= 10  ------------
 rep 0:  tr_time=2.8903 acc_tr=0.9188, acc_tst=0.8980, n_large_cf=1713
 rep 1:  tr_time=2.9014 acc_tr=0.9188, acc_tst=0.8959, n_large_cf=1733
 rep 2:  tr_time=2.9068 acc_tr=0.9196, acc_tst=0.8976, n_large_cf=1709
tr_time: [mean=2.8995, std=0.0069],  acc_tr: [mean=0.9191, std=0.0004], acc_tst: [mean=0.8972, std=0.0009]
------- C= 0.005  n_splits= 20  ------------
 rep 0:  tr_time=2.5600 acc_tr=0.9198, acc_tst=0.8962, n_large_cf=1803
 rep 1:  tr_time=2.7691 acc_tr=0.9191, acc_tst=0.8978, n_large_cf=1781
 rep 2:  tr_time=2.7998 acc_tr=0.9183, acc_tst=0.8976, n_large_cf=1786
tr_time: [mea

In [14]:
# Усреднение несольких объединенных гиперплоскостей 
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test10k
Ytst = Y_test10k 

thr = 0.0000001
nreps = 10
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
tr_times = np.zeros((nreps))
tr_accs = np.zeros((nreps))
tst_accs = np.zeros((nreps))

#splits = [5, 10, 20, 50, 100, 200, 500, 1000]
splits = [5, 1000]
#cs = [0.005, 0.0075, 0.01, 0.025, 0.05, 0.075, 0.1]  
cs = [0.005, 0.1]  
for C in cs:     
    for n_splits in splits:
        print('------- C=',C,' n_splits=',n_splits,' ------------')
        for rep in range(nreps):
            tr_res = dict()
            model = LinearSVC(C=C, penalty="l1", dual=False)
            tr_res = ConstractCombinedHyperplane(Xtr, Ytr, model=model, n_splits=n_splits)
            
            # test combined hyperplane
            big_a = (abs(tr_res['comb']['a'])>thr)
            comb_inds = tr_res['comb']['inds']
            large_inds = comb_inds[big_a]
            short_model=dict()
            short_model['a'] = tr_res['comb']['a'][big_a]
            short_model['b'] = tr_res['comb']['b']
            short_model['inds'] = large_inds
            acc_tr, _ = TestLinearSVM_v2(Xtr[:,large_inds],Ytr, short_model) 
            acc_tst, _ = TestLinearSVM_v2(Xtst[:,large_inds],Ytst, short_model)

            tr_times[rep] = tr_res['comb']['full_time']
            tr_accs[rep] = acc_tr
            tst_accs[rep] = acc_tst
            print(f" rep {rep}:  tr_time={tr_times[rep]:.4f} acc_tr={acc_tr:.4f}, acc_tst={acc_tst:.4f}, n_large_cf={len(large_inds)}")

            # construct mean hyperplane
            inds = tr_res['comb']['inds']
            a = tr_res['comb']['a'] 
            b = tr_res['comb']['b']
            tm = tr_res['comb']['full_time']
            if rep==0:
                sum_model = dict()
                sum_model['a'] = np.zeros((n_feat))
                sum_model['a'][inds] = a 
                sum_model['b'] = b
                sum_model['time'] = tm
            else: 
                sum_model['a'][inds] = sum_model['a'][inds] + a 
                sum_model['b'] = sum_model['b'] + b
                sum_model['time'] = sum_model['time'] + tm
            
            mean_model = dict()    
            mean_model['a'] = sum_model['a']/(rep+1)
            mean_model['b'] = sum_model['b']/(rep+1) 
            mean_model['time'] = sum_model['time']

            # test mean hyperplane
            acc_tr_mean, _ = TestLinearSVM_v2(Xtr,Ytr, mean_model) 
            acc_tst_mean, _ = TestLinearSVM_v2(Xtst,Ytst, mean_model)
            
            print(f" mean hyperplane: time = {mean_model['time']:.4f},  acc_tr_mean ={acc_tr_mean:.4f}, acc_tst_mean ={acc_tst_mean:.4f}")
        print(f"tr_time: [mean={np.mean(tr_times):.4f}, std={np.std(tr_times):.4f}],  acc_tr: [mean={np.mean(tr_accs):.4f}, std={np.std(tr_accs):.4f}], acc_tst: [mean={np.mean(tst_accs):.4f}, std={np.std(tst_accs):.4f}]")
    

n_obj =  10000  n_feat= 5000
------- C= 0.005  n_splits= 5  ------------
 rep 0:  tr_time=2.8282 acc_tr=0.9206, acc_tst=0.8973, n_large_cf=1787
 mean hyperplane: time = 2.8282,  acc_tr_mean =0.9206, acc_tst_mean =0.8973
 rep 1:  tr_time=2.9987 acc_tr=0.9196, acc_tst=0.8986, n_large_cf=1802
 mean hyperplane: time = 5.8269,  acc_tr_mean =0.9198, acc_tst_mean =0.8985
 rep 2:  tr_time=2.9200 acc_tr=0.9200, acc_tst=0.8976, n_large_cf=1785
 mean hyperplane: time = 8.7468,  acc_tr_mean =0.9198, acc_tst_mean =0.8994
 rep 3:  tr_time=3.0006 acc_tr=0.9202, acc_tst=0.8970, n_large_cf=1811
 mean hyperplane: time = 11.7474,  acc_tr_mean =0.9210, acc_tst_mean =0.8997
 rep 4:  tr_time=3.0990 acc_tr=0.9204, acc_tst=0.8993, n_large_cf=1827
 mean hyperplane: time = 14.8464,  acc_tr_mean =0.9207, acc_tst_mean =0.9003
 rep 5:  tr_time=2.7333 acc_tr=0.9177, acc_tst=0.8976, n_large_cf=1796
 mean hyperplane: time = 17.5797,  acc_tr_mean =0.9206, acc_tst_mean =0.8999
 rep 6:  tr_time=3.0003 acc_tr=0.9207, acc

In [15]:
%%time
# l1-based feature selection
#Linear models penalized with the L1 norm have sparse solutions: many of their estimated coefficients are zero. 
# When the goal is to reduce the dimensionality of the data to use with another classifier, they can be used along with SelectFromModel 
#to select the non-zero coefficients. In particular, sparse estimators useful for this purpose are the Lasso for regression, 
#and of LogisticRegression and LinearSVC for classification
# With SVMs and logistic regression, the parameter C controls the sparsity: the smaller C the fewer features selected
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test10k
Ytst = Y_test10k 
n_obj, n_feat = np.shape(Xtr)
print('LinearSVC, l1 penalty')
print('n_obj = ', n_obj, ' n_feat=', n_feat)
#cs = [0.005, 0.0075, 0.01, 0.025, 0.05, 0.075, 0.1, 0.2, 0.3, 0.4, 0.5]  
cs = [0.005, 0.0075, 0.01, 0.025, 0.05]  
for C in cs:
    model = LinearSVC(C=C, penalty="l1", dual=False)
    model = TrainLinearSVM_v2(Xtr, Ytr, model) 
    inds = np.array(range(n_feat))
    acc_tr, _ = TestLinearSVM_v2(Xtr,Ytr, model)
    acc_tst, _ = TestLinearSVM_v2(Xtst,Ytst, model)
    #_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
    #_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)
    print(f"C={C:.4f}: tr_time={model['time']:.4f}, acc_tr = {acc_tr:.4f}', acc_tst={acc_tst:.4f}")

LinearSVC, l1 penalty
n_obj =  10000  n_feat= 5000
C=0.0050: tr_time=3.2032, acc_tr = 0.9184', acc_tst=0.9000
C=0.0075: tr_time=3.5304, acc_tr = 0.9282', acc_tst=0.9009
C=0.0100: tr_time=4.2082, acc_tr = 0.9378', acc_tst=0.8987
C=0.0250: tr_time=11.0037, acc_tr = 0.9732', acc_tst=0.8643
C=0.0500: tr_time=96.1790, acc_tr = 0.9944', acc_tst=0.8224
C=0.1000: tr_time=241.8137, acc_tr = 1.0000', acc_tst=0.7975
CPU times: total: 5min 52s
Wall time: 6min 2s


In [ ]:
# предварительный отбор признаков 

In [16]:
#Random Subspace Ensemble via Bagging
#https://machinelearningmastery.com/random-subspace-ensemble-with-python/ 

from sklearn.ensemble import BaggingClassifier 
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test10k
Ytst = Y_test10k 

#nfs = [10, 50, 100, 500, 1000]
nfs = [50, 100, 500]
#nests = [10, 50, 100, 500, 1000]
nests = [10, 100, 500]
#cs = [0.01, 0.1, 0.5] 
cs = [0.005, 0.01, 0.1] 
for C in cs:
    print('------- C=', C, ' ---------')
    estimator = LinearSVC(C=C, penalty="l1", dual=False)
    for max_feats in nfs:
        for n_est in nests:
            time_start = time.time()
            model_bag = BaggingClassifier(estimator=estimator, bootstrap=False, max_features=max_feats, n_estimators=n_est, random_state=42)
            model_bag.fit(Xtr, Ytr)
            time_end = time.time()
            tr_time = time_end - time_start
            pr_tr = model_bag.predict(Xtr)
            pr_tst = model_bag.predict(Xtst)
            accTr = accuracy_score(pr_tr, Ytr)
            accTst = accuracy_score(pr_tst, Ytst)
            print(f"max_feats={max_feats}, n_ests={n_est}, Bagging accuracy: acc_tr={accTr:.4f} acc_tst={accTst:.4f} tr_time={tr_time:.4f}")


------- C= 0.005  ---------
max_feats=50, n_ests=10, Bagging accuracy: acc_tr=0.6060 acc_tst=0.5923 tr_time=0.3009
max_feats=50, n_ests=100, Bagging accuracy: acc_tr=0.7850 acc_tst=0.7423 tr_time=2.9636
max_feats=50, n_ests=500, Bagging accuracy: acc_tr=0.8938 acc_tst=0.8483 tr_time=14.3912
max_feats=100, n_ests=10, Bagging accuracy: acc_tr=0.6534 acc_tst=0.6255 tr_time=0.5562
max_feats=100, n_ests=100, Bagging accuracy: acc_tr=0.8453 acc_tst=0.7927 tr_time=5.6711
max_feats=100, n_ests=500, Bagging accuracy: acc_tr=0.9058 acc_tst=0.8726 tr_time=28.7885
max_feats=500, n_ests=10, Bagging accuracy: acc_tr=0.8008 acc_tst=0.7478 tr_time=2.9289
max_feats=500, n_ests=100, Bagging accuracy: acc_tr=0.9118 acc_tst=0.8861 tr_time=29.7025
max_feats=500, n_ests=500, Bagging accuracy: acc_tr=0.9177 acc_tst=0.8968 tr_time=146.9786
------- C= 0.01  ---------
max_feats=50, n_ests=10, Bagging accuracy: acc_tr=0.6126 acc_tst=0.5920 tr_time=0.3306
max_feats=50, n_ests=100, Bagging accuracy: acc_tr=0.7951 

In [ ]:
# надо переделывать! 
def RFEselection(X_train, Y_train, nf_sel):
#Feature ranking with recursive feature elimination
#The number of features to select. If None, half of the features are selected. 
#If integer, the parameter is the absolute number of features to select. 
#If float between 0 and 1, it is the fraction of features to select
    X_train_df=pd.DataFrame(X_train)
    #from sklearn.feature_selection import RFE
    #svc_lin=SVC(kernel='linear')
    svc_lin=LinearSVC(C=0.01, penalty="l1", dual=False)
    svm_rfe_model=RFE(estimator=svc_lin, n_features_to_select = nf_sel, step = 1)
    svm_rfe_model_fit=svm_rfe_model.fit(X_train_df,Y_train)
    feat_index = pd.Series(data = svm_rfe_model_fit.ranking_)
    signi_feat_rfe = feat_index[feat_index==1].index # indexes of selected features
    #print('Significant features from RFE',signi_feat_rfe)
    #print(svm_rfe_model_fit.ranking_)
    #print('score = ', svm_rfe_model_fit.score(X_trains_df, Y_train))
    sel_feat = svm_rfe_model_fit.transform(X_train_df) # submatrix of selected features
    #score = svm_rfe_model_fit.score(X_train_df, Y_train)  
    return signi_feat_rfe, feat_index, svm_rfe_model_fit.ranking_, sel_feat 

def RFEselection_and_find_acc(X_trains, Y_train, X_vals, Y_val, X_tests, Y_test, n_sel_feat):
    time_start = time.time()
    selfeat_inds, feat_ranks, ranking, X_selfeat = RFEselection(X_trains, Y_train, n_sel_feat)
    svc_selfeat=SVC(kernel='linear')
    svc_selfeat.fit(X_selfeat,Y_train)
    time_end_train = time.time() 
    print('train_time =', time_end_train-time_start)
    y_pred_train = svc_selfeat.predict(X_selfeat)
    acc_tr = accuracy_score(Y_train, y_pred_train)
    #print(selfeat_inds)
    X_selfeat_test = X_tests[:,selfeat_inds]
    y_pred_test = svc_selfeat.predict(X_selfeat_test)
    acc_tst = accuracy_score(Y_test, y_pred_test)

    X_selfeat_val = X_vals[:,selfeat_inds]
    y_pred_val = svc_selfeat.predict(X_selfeat_val)
    acc_val = accuracy_score(Y_val, y_pred_val)

    print('n_sel_feat=',n_sel_feat, 'selfeat_inds:',selfeat_inds)
    print('accuracy train:', acc_tr, 'accuracy test:', acc_tst, 'accuracy val:', acc_val)
    return selfeat_inds, acc_tr, acc_tst, acc_val, time_end_train-time_start

In [ ]:
## Initialize the SVM with class_weight='balanced'
svm_model = SVC(class_weight='balanced', random_state=42)
# Method 2: Manual Class Weighting 
# Manually defining weights (e.g., class 0 has weight 1, class 1 has weight 10)
weights = {0: 1, 1: 10}

svm_model_manual = SVC(class_weight=weights, random_state=42)
svm_model_manual.fit(X_train, y_train)

In [ ]:
#This example will also work by replacing SVC(kernel="linear") with SGDClassifier(loss="hinge"). Setting the loss parameter of the SGDClassifier equal to hinge will yield behaviour such as that of an SVC with a linear kernel.

#For example try instead of the SVC:
clf = SGDClassifier(n_iter=100, alpha=0.01) 


In [ ]:
#SVM-Anova: SVM with univariate feature selection 
#https://scikit-learn.org/stable/auto_examples/svm/plot_svm_anova.html 

In [ ]:
# Univariate Feature Selection 
#https://scikit-learn.org/stable/auto_examples/feature_selection/plot_feature_selection.html